# RSNA Knee — Preprocessing (224 px)

Your original preprocessing pipeline, unchanged except for the output resolution.

**The only change: `OUT_SIZE` 192 → 224.**

```
v1:  160 mm / 192 px = 0.833 mm/pixel
now: 160 mm / 224 px = 0.714 mm/pixel
```

224 is chosen because it is divisible by both **16** (DINOv3 patch size) and **14** (DINOv2), so the same
cache feeds either backbone with no resampling inside the model.

Everything else is identical: the same five cells, the same 160 mm crop, the same slice ordering by
projection onto the slice normal, the same laterality resolution (tag → report → geometry), the same
per-series intensity normalisation, the same fallback rules.

### One thing to watch: cache size

```
4407 studies x 104 slices x 224^2 = 23.0 GB uncompressed   (v1 at 192 px: 16.9 GB)
```

`np.savez_compressed` typically brings that to 13–16 GB on disk, which fits `/kaggle/working`. §7 projects
the real number from a pilot **before** the full run and stops if it would overflow — a cache that
overshoots kills the kernel rather than slowing it down.

If the gate trips, lower the per-cell slice counts in `CFG["CELLS"]` (the 4th field of each tuple).

### A note on slice counts, since you train with 21

The cells hold 24 / 24 / 20 / 20 / 16 slices. Your training notebook takes `N_SLICES_PER_VIEW = 21` centred
slices, so for `COR_FS`, `COR_T1` (20) and `AX_FS` (16) the index picker clamps at the stack edges and
repeats the outermost slice to reach 21. That is your existing v1 behaviour and it is preserved here
unchanged — but if you want 21 genuinely distinct slices everywhere, raise those three counts to 24 (cache
grows to ~26.5 GB raw, so check the gate).


In [ ]:
import os, sys, re, gc, json, time, math, glob, unicodedata, warnings, traceback
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom, cv2

def find_root():
    for c in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
              "/kaggle/input/rsna-knee-abnormality-detection"]:
        if os.path.exists(os.path.join(c, "train.csv")): return c
    for p in sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)):
        return os.path.dirname(p)
    raise FileNotFoundError("train.csv not found")

DATA = find_root()
OUT  = Path("/kaggle/working/prep_out"); (OUT / "cache").mkdir(parents=True, exist_ok=True)

LABELS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
          "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

CFG = {
    # ---- spatial normalisation ------------------------------------------------
    "OUT_SIZE": 224,          # CHANGED from 192. 160mm/224px = 0.714 mm/px
    "FOV_MM": 160.0,
    # ---- the 5 canonical (plane, fluid, fatsat) cells -------------------------
    # name        plane       fluid fatsat  n_slices  extent_mm   fallback cell
    "CELLS": [
        ("SAG_FS", "Sagittal", 1, 1, 24, 92.0, "SAG_T1"),
        ("SAG_T1", "Sagittal", 0, 0, 24, 92.0, "SAG_FS"),
        ("COR_FS", "Coronal",  1, 1, 21, 76.0, "COR_T1"),
        ("COR_T1", "Coronal",  0, 0, 21, 76.0, "COR_FS"),
        ("AX_FS",  "Axial",    1, 1, 21, 96.0, None),
    ],
    # ---- intensity ------------------------------------------------------------
    "CLIP_LO_PCT": 0.5,
    "CLIP_HI_PCT": 99.5,
    "FG_THRESH_FRAC": 0.06,
    # ---- laterality -----------------------------------------------------------
    "CANONICAL_SIDE": "R",
    "LAT_GEOM_THRESH": None,
    # ---- run control ----------------------------------------------------------
    "N_WORKERS": 4,
    "PILOT_N": 40,
    "COMPRESS": True,
    "CACHE_GB_BUDGET": 18.0,
    "SEED": 42,
}
np.random.seed(CFG["SEED"])

CELL_NAMES = [c[0] for c in CFG["CELLS"]]
CELL_BY_NAME = {c[0]: dict(zip(["name","plane","fluid","fatsat","n_slices","extent_mm","fallback"], c))
                for c in CFG["CELLS"]}

print("DATA:", DATA)
print("mm per pixel  :", round(CFG["FOV_MM"] / CFG["OUT_SIZE"], 4), "  (v1 at 192px: 0.8333)")
print("cells         :", CELL_NAMES)
print("slices/study  :", sum(c[4] for c in CFG["CELLS"]))
print("raw bytes/study:", f"{sum(c[4] for c in CFG['CELLS']) * CFG['OUT_SIZE']**2 / 1e6:.2f} MB")


## 1. Decoder check

Training data appears to be entirely uncompressed, so a missing JPEG/JPEG2000 codec can only surface in the
hidden test set — the worst possible place to discover it.


In [ ]:
def decoder_status():
    st = {}
    for m in ["pylibjpeg", "libjpeg", "openjpeg", "gdcm"]:
        try: __import__(m); st[m] = True
        except Exception: st[m] = False
    return st

DEC = decoder_status(); print(DEC)
if not (DEC["gdcm"] or (DEC["libjpeg"] and DEC["openjpeg"])):
    print("\n" + "!"*78)
    print("NO JPEG/JPEG2000 DECODER AVAILABLE.")
    print("Uncompressed DICOMs still work so this notebook will finish, but compressed series")
    print("in the hidden test set will fail. Attach codec wheels before submitting:")
    print("  pip install --no-index --find-links=<dataset> pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg")
    print("!"*78)


## 2. Tables and series selection

Selection ignores `SeriesDescription` (~21% are `DUMMYSERIESDESC!` or blank) and ranks on slice count in log
space: a knee series has ~30 slices, so this demotes localisers and dynamic acquisitions without hard-coded
thresholds. Ties break on the smallest UID so the choice is reproducible.


In [ ]:
train = pd.read_csv(os.path.join(DATA, "train.csv"))
tser  = pd.read_csv(os.path.join(DATA, "train_series.csv"))
test  = pd.read_csv(os.path.join(DATA, "test.csv"))
tests = pd.read_csv(os.path.join(DATA, "test_series.csv"))
print(train.shape, tser.shape, test.shape, tests.shape)

def count_slices(root, study, series):
    try: return sum(1 for f in os.scandir(os.path.join(root, study, series))
                    if f.name.endswith(".dcm"))
    except FileNotFoundError: return 0

def add_slice_counts(df, root):
    with ThreadPoolExecutor(max_workers=16) as ex:
        n = list(ex.map(lambda r: count_slices(root, r[0], r[1]),
                        df[["StudyInstanceUID","SeriesInstanceUID"]].values))
    out = df.copy(); out["n_slices"] = n; return out

ROOT_TR = os.path.join(DATA, "train_series")
t0 = time.time(); tser = add_slice_counts(tser, ROOT_TR)
print(f"slice counts in {time.time()-t0:.0f}s | empty series: {(tser['n_slices']==0).sum()}")
print(tser["n_slices"].describe().round(1).to_string())


In [ ]:
def cell_key(row):
    for name, plane, fl, fs, *_ in CFG["CELLS"]:
        if row["Anatomical_Plane"] == plane and int(row["Fluid_Sensitive"]) == fl \
           and int(row["Fat_Suppression"]) == fs:
            return name
    return None

def build_series_index(series_df):
    df = series_df.copy()
    df["cell"] = df.apply(cell_key, axis=1)
    df = df[df["cell"].notna() & (df["n_slices"] > 0)].copy()
    df["score"] = -np.abs(np.log(df["n_slices"].clip(lower=1) / 30.0))
    df = df.sort_values(["StudyInstanceUID","cell","score","SeriesInstanceUID"],
                        ascending=[True, True, False, True])
    return df, df.groupby(["StudyInstanceUID","cell"], as_index=False).first()

all_cells, SEL = build_series_index(tser)
print("chosen rows:", len(SEL))
piv = SEL.pivot_table(index="StudyInstanceUID", columns="cell",
                      values="n_slices", aggfunc="first").reindex(columns=CELL_NAMES)
print("\ncell availability after selection:")
for c in CELL_NAMES:
    print(f"  {c:<10} {100*piv[c].notna().mean():5.1f} %")
print("\ncells per study:")
print(piv.notna().sum(axis=1).value_counts().sort_index().to_string())


## 3. Reports and laterality

Four of the twelve labels are medial/lateral pairs and a fifth (MCL) is named for its side, so mirroring
every knee into one convention is the transform that matters most. Resolution order: DICOM `Laterality` tag
→ report text → geometry, with the geometry threshold **fitted and accuracy-reported** on the tagged half.


In [ ]:
PLACEHOLDER = re.compile(r"\[(DATE|TIME|ID|NAME|AGE|PHONE|ADDRESS|DATETIME|MRN|ACCESSION)\]", re.I)
WS = re.compile(r"[ \t\u00a0]+"); NL3 = re.compile(r"\n{3,}")

def clean_report(t):
    if not isinstance(t, str): return ""
    t = unicodedata.normalize("NFKC", t); t = PLACEHOLDER.sub(" ", t); t = t.replace("\r", "\n")
    return NL3.sub("\n\n", WS.sub(" ", t)).strip()

def script_of(t):
    if re.search(r"[\u0370-\u03ff\u1f00-\u1fff]", t): return "greek"
    if re.search(r"[\u0400-\u04ff]", t): return "cyrillic"
    if re.search(r"[\u0600-\u06ff]", t): return "arabic"
    return "latin"

STOP = {"en":{"the","and","with","there","normal","tear","joint","signal","findings","knee"},
        "es":{"el","la","los","con","del","sin","rodilla","articular","menisco","hallazgos"},
        "fr":{"le","les","des","avec","sans","genou","articulaire","menisque","aspect"},
        "de":{"der","die","das","und","mit","ohne","kniegelenk","kein","nachweis","befund"},
        "it":{"il","del","della","con","senza","ginocchio","menisco","articolare","non"},
        "pt":{"do","da","com","sem","joelho","menisco","articular","nao","em"},
        "nl":{"de","het","een","met","zonder","knie","geen","van","bij","links"},
        "tr":{"ve","ile","yok","diz","eklem","izlendi","olan","mevcut","sinyal"}}
def guess_lang(t):
    s = script_of(t)
    if s != "latin": return s
    toks = set(re.findall(r"[a-zA-Zçğşöüıáéíóúñäöüß]+", t.lower()))
    best, n = "unk", 0
    for k, v in STOP.items():
        m = len(toks & v)
        if m > n: best, n = k, m
    return best if n >= 2 else "unk"

LAT_PAT = {"R": r"\b(right|rt\.?|derech[ao]|direita|droit[e]?|rechts?|destr[ao]|sağ|sag\b|правый|правая|справа|δεξι)",
           "L": r"\b(left|lt\.?|izquierd[ao]|esquerda|gauche|links?|sinistr[ao]|sol\b|левый|левая|слева|αριστερ)"}
def lat_from_report(t):
    tl = t.lower()
    r_, l_ = len(re.findall(LAT_PAT["R"], tl)), len(re.findall(LAT_PAT["L"], tl))
    return "R" if r_ > l_ else ("L" if l_ > r_ else None)

REP = pd.DataFrame({"StudyInstanceUID": train["StudyInstanceUID"]})
REP["report_raw"]   = train["Report"].fillna("")
REP["report_clean"] = REP["report_raw"].map(clean_report)
REP["lang"]         = REP["report_clean"].map(guess_lang)
REP["n_chars"]      = REP["report_clean"].str.len()
REP["lat_report"]   = REP["report_clean"].map(lat_from_report)
if "PatientSex" in train.columns: REP["PatientSex"] = train["PatientSex"]
print(REP["lang"].value_counts().to_string())
print("\nreport laterality found:", round(REP["lat_report"].notna().mean(), 3))


In [ ]:
def study_lat_probe(root, study, series_ids):
    tags, xs = [], []
    for se in list(series_ids)[:3]:
        d = os.path.join(root, study, se)
        try: files = sorted(f.name for f in os.scandir(d) if f.name.endswith(".dcm"))
        except FileNotFoundError: continue
        if not files: continue
        for f in [files[0], files[len(files)//2]]:
            try: ds = pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True, force=True)
            except Exception: continue
            lt = str(getattr(ds, "Laterality", "") or "").strip().upper()
            if lt in ("L","R"): tags.append(lt)
            iop = getattr(ds, "ImageOrientationPatient", None)
            ipp = getattr(ds, "ImagePositionPatient", None)
            ps  = getattr(ds, "PixelSpacing", None)
            cols, rows = getattr(ds, "Columns", None), getattr(ds, "Rows", None)
            if iop is None or ipp is None or ps is None: continue
            iop = np.array(iop, float); ipp = np.array(ipp, float); ps = np.array(ps, float)
            centre = ipp + iop[:3]*(float(cols)/2*ps[1]) + iop[3:]*(float(rows)/2*ps[0])
            xs.append(float(centre[0]))
    return (Counter(tags).most_common(1)[0][0] if tags else None,
            float(np.mean(xs)) if xs else np.nan)

studies = SEL["StudyInstanceUID"].unique()
ser_by_study = SEL.groupby("StudyInstanceUID")["SeriesInstanceUID"].apply(list).to_dict()
t0 = time.time()
with ThreadPoolExecutor(max_workers=12) as ex:
    res = list(ex.map(lambda s: study_lat_probe(ROOT_TR, s, ser_by_study[s]), studies))
print(f"probed {len(studies)} studies in {time.time()-t0:.0f}s")

LAT = pd.DataFrame({"StudyInstanceUID": studies,
                    "lat_tag": [r[0] for r in res], "centre_x": [r[1] for r in res]})
LAT = LAT.merge(REP[["StudyInstanceUID","lat_report"]], on="StudyInstanceUID", how="left")
print("tag available   :", round(LAT["lat_tag"].notna().mean(), 3))
print("report available:", round(LAT["lat_report"].notna().mean(), 3))

both = LAT.dropna(subset=["lat_tag","lat_report"])
print(f"\nreport vs tag agreement: {(both['lat_tag']==both['lat_report']).mean()*100:.1f}% (n={len(both)})")

g = LAT.dropna(subset=["lat_tag","centre_x"])
cand = np.arange(-200, 100, 1.0)
best_acc, best_t = max(((np.where(g["centre_x"].values > t, "L", "R") == g["lat_tag"].values).mean(), t)
                       for t in cand)
CFG["LAT_GEOM_THRESH"] = float(best_t)
print(f"geometry rule: centre_x > {best_t:.1f} mm => LEFT | accuracy {best_acc*100:.2f}% (n={len(g)})")

plt.figure(figsize=(8, 3.2))
for k, grp in g.groupby("lat_tag"):
    plt.hist(grp["centre_x"], bins=60, alpha=0.6, label=f"tag={k}")
plt.axvline(best_t, color="k", ls="--", label=f"threshold {best_t:.0f}")
plt.legend(); plt.xlabel("image centre x in patient space (mm)")
plt.title("Geometry-based laterality separation"); plt.tight_layout(); plt.show()

GEOM_MIN_ACC = 0.95
def resolve_lat(row):
    if isinstance(row["lat_tag"], str):    return row["lat_tag"], "tag"
    if isinstance(row["lat_report"], str): return row["lat_report"], "report"
    if best_acc >= GEOM_MIN_ACC and row["centre_x"] == row["centre_x"]:
        return ("L" if row["centre_x"] > CFG["LAT_GEOM_THRESH"] else "R"), "geometry"
    return "U", "unknown"

r_ = LAT.apply(resolve_lat, axis=1, result_type="expand")
LAT["laterality"], LAT["lat_source"] = r_[0], r_[1]
print("\n" + LAT["lat_source"].value_counts().to_string())
print(LAT["laterality"].value_counts().to_string())
print(f"studies to be mirrored: {(LAT['laterality'] == ('L' if CFG['CANONICAL_SIDE']=='R' else 'R')).sum()}")


## 4. The transform

Unchanged from v1. Slices are ordered by projection onto the slice normal — never by filename, since the
filename is a SOP Instance UID assigned to be unique rather than ordered, and your v1 EDA measured
`InstanceNumber` agreeing with geometric order only 63.8% of the time.


In [ ]:
def _read_geom(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
    iop = getattr(ds, "ImageOrientationPatient", None)
    ipp = getattr(ds, "ImagePositionPatient", None)
    ps  = getattr(ds, "PixelSpacing", None)
    if iop is None or ipp is None or ps is None: return None
    return dict(path=path, iop=np.array(iop, float), ipp=np.array(ipp, float),
                ps=np.array(ps, float), rows=int(ds.Rows), cols=int(ds.Columns),
                slope=float(getattr(ds, "RescaleSlope", 1) or 1),
                inter=float(getattr(ds, "RescaleIntercept", 0) or 0),
                photo=str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")))

def _pixels(g):
    ds = pydicom.dcmread(g["path"], force=True)
    a = ds.pixel_array.astype(np.float32)
    if g["slope"] != 1 or g["inter"] != 0: a = a * g["slope"] + g["inter"]
    if g["photo"].upper() == "MONOCHROME1": a = a.max() - a
    np.clip(a, 0, None, out=a)
    return a

def _resample_crop(a, ps_rc, out_size, mm_per_px, centre_rc=None):
    sh = max(1, int(round(a.shape[0] * ps_rc[0] / mm_per_px)))
    sw = max(1, int(round(a.shape[1] * ps_rc[1] / mm_per_px)))
    interp = cv2.INTER_AREA if (sh < a.shape[0] or sw < a.shape[1]) else cv2.INTER_LINEAR
    r = cv2.resize(a, (sw, sh), interpolation=interp)
    if centre_rc is None: centre_rc = (sh/2.0, sw/2.0)
    r0 = int(round(centre_rc[0] - out_size/2)); c0 = int(round(centre_rc[1] - out_size/2))
    out = np.zeros((out_size, out_size), np.float32)
    sr0, sc0 = max(0, r0), max(0, c0)
    sr1, sc1 = min(sh, r0+out_size), min(sw, c0+out_size)
    if sr1 > sr0 and sc1 > sc0:
        out[sr0-r0:sr1-r0, sc0-c0:sc1-c0] = r[sr0:sr1, sc0:sc1]
    return out

def _fg_centroid(a, frac):
    lo, hi = float(a.min()), float(np.percentile(a, 99.5))
    m = a > lo + frac*(hi-lo)
    if m.sum() < 64: return None
    rs, cs = np.nonzero(m)
    return (float(rs.mean()), float(cs.mean()))

def load_cell(root, study, series, cell, laterality, cfg=CFG):
    spec = CELL_BY_NAME[cell]
    d = os.path.join(root, study, series)
    files = sorted(f.path for f in os.scandir(d) if f.name.endswith(".dcm"))
    if not files: raise RuntimeError("no dcm files")

    geoms = [g for g in (_read_geom(p) for p in files) if g is not None]
    if len(geoms) < 2: raise RuntimeError("insufficient geometry")

    iop0 = geoms[0]["iop"]
    n = np.cross(iop0[:3], iop0[3:]); n /= (np.linalg.norm(n) + 1e-9)
    t = np.array([float(np.dot(g["ipp"], n)) for g in geoms])
    order = np.argsort(t); geoms = [geoms[i] for i in order]; t = t[order]

    keep = [0] + [i for i in range(1, len(t)) if abs(t[i]-t[i-1]) > 1e-3]
    geoms = [geoms[i] for i in keep]; t = t[keep]
    n_dup = len(order) - len(keep)

    N, ext = spec["n_slices"], spec["extent_mm"]
    span = float(t[-1] - t[0])
    if span >= ext and len(t) > 1:
        centre = 0.5*(t[0]+t[-1]); want = centre + np.linspace(-ext/2, ext/2, N)
        step_mm = ext / max(N-1, 1)
    else:
        want = np.linspace(t[0], t[-1], N); step_mm = span / max(N-1, 1)
    idx = [int(np.argmin(np.abs(t - w))) for w in want]

    mm = cfg["FOV_MM"] / cfg["OUT_SIZE"]
    mid = geoms[idx[len(idx)//2]]
    cen = _fg_centroid(_pixels(mid), cfg["FG_THRESH_FRAC"])
    if cen is not None: cen = (cen[0]*mid["ps"][0]/mm, cen[1]*mid["ps"][1]/mm)

    vol = np.empty((N, cfg["OUT_SIZE"], cfg["OUT_SIZE"]), np.float32)
    for k, i in enumerate(idx):
        g = geoms[i]
        vol[k] = _resample_crop(_pixels(g), g["ps"], cfg["OUT_SIZE"], mm, cen)

    mirrored = False
    if laterality in ("L","R") and laterality != cfg["CANONICAL_SIDE"]:
        vol = np.flip(vol, axis=0) if spec["plane"] == "Sagittal" else np.flip(vol, axis=2)
        vol = np.ascontiguousarray(vol); mirrored = True

    lo0, hi0 = float(vol.min()), float(np.percentile(vol, 99.5))
    fg = vol[vol > lo0 + cfg["FG_THRESH_FRAC"]*(hi0-lo0)]
    if fg.size < 1000: fg = vol.reshape(-1)
    lo = float(np.percentile(fg, cfg["CLIP_LO_PCT"])); hi = float(np.percentile(fg, cfg["CLIP_HI_PCT"]))
    if hi - lo < 1e-6: hi = lo + 1.0
    vol = np.clip((vol - lo)/(hi - lo), 0, 1)
    vol = (vol*255.0 + 0.5).astype(np.uint8)

    meta = dict(cell=cell, series=series, n_src_slices=len(t), span_mm=round(span,2),
                step_mm=round(float(step_mm),3), px_mm_src=round(float(geoms[0]["ps"][0]),4),
                src_rows=geoms[0]["rows"], src_cols=geoms[0]["cols"],
                mirrored=mirrored, n_dup=n_dup, short_stack=bool(span < ext))
    return vol, meta


In [ ]:
def process_study(args):
    study, rows, laterality, root, cfg = args
    out, meta, errs, have = {}, {}, {}, {}
    for cell in CELL_NAMES:
        se = rows.get(cell)
        if se is None: continue
        try:
            v, m = load_cell(root, study, se, cell, laterality, cfg)
            out[cell], meta[cell], have[cell] = v, m, 1
        except Exception as e:
            errs[cell] = f"{type(e).__name__}: {e}"[:160]
    for cell in CELL_NAMES:
        if cell in out: continue
        fb = CELL_BY_NAME[cell]["fallback"]
        spec = CELL_BY_NAME[cell]
        # `have[fb] == 1` (not `fb in out`): a cell zero-filled earlier in THIS loop is also
        # "in out", and copying from it would flag an all-zero cell as available -- and would
        # KeyError on meta[fb], which is only written in the try block above.
        if fb and have.get(fb) == 1 and out[fb].shape[0] == spec["n_slices"]:
            out[cell] = out[fb].copy(); have[cell] = 2
            meta[cell] = dict(meta[fb], cell=cell, filled_from=fb)
        else:
            out[cell] = np.zeros((spec["n_slices"], cfg["OUT_SIZE"], cfg["OUT_SIZE"]), np.uint8)
            have[cell] = 0
    return study, out, meta, have, errs

def save_study(study, out, have, cfg=CFG):
    p = OUT / "cache" / f"{study}.npz"
    payload = {c: out[c] for c in CELL_NAMES}
    payload["avail"] = np.array([have[c] for c in CELL_NAMES], np.uint8)
    (np.savez_compressed if cfg["COMPRESS"] else np.savez)(p, **payload)
    return p.stat().st_size

lat_map = dict(zip(LAT["StudyInstanceUID"], LAT["laterality"]))
grp = SEL.groupby("StudyInstanceUID").apply(
    lambda d: dict(zip(d["cell"], d["SeriesInstanceUID"]))).to_dict()
JOBS = [(s, rows, lat_map.get(s, "U"), ROOT_TR, CFG) for s, rows in grp.items()]
print("jobs:", len(JOBS))


## 5. Pilot and size gate


In [ ]:
pilot = JOBS[:CFG["PILOT_N"]]
t0 = time.time(); sizes = []; shown = {}; pilot_err = []
for j in pilot:
    st, out, meta, have, errs = process_study(j)
    sizes.append(save_study(st, out, have))
    for c, e in errs.items(): pilot_err.append((st, c, e))
    if len(shown) < 2: shown[st] = (out, have)
dt = time.time() - t0
proj_gb = float(np.mean(sizes))*len(JOBS)/1e9
print(f"{len(pilot)} studies in {dt:.1f}s -> {dt/len(pilot)*1000:.0f} ms/study")
print(f"mean npz {np.mean(sizes)/1e6:.2f} MB | projected cache {proj_gb:.1f} GB "
      f"(budget {CFG['CACHE_GB_BUDGET']} GB)")
print(f"projected wall time at {CFG['N_WORKERS']} workers: "
      f"{dt/len(pilot)*len(JOBS)/CFG['N_WORKERS']/60:.1f} min")
if pilot_err:
    print("\nerrors:"); [print("  ", e) for e in pilot_err[:8]]
else:
    print("\nno errors in pilot")

if proj_gb > CFG["CACHE_GB_BUDGET"]:
    raise RuntimeError(
        f"projected cache {proj_gb:.1f} GB exceeds the {CFG['CACHE_GB_BUDGET']} GB budget. "
        f"Lower the per-cell slice counts in CFG['CELLS'] and re-run. Overshooting "
        f"/kaggle/working kills the kernel rather than slowing it.")
print("size gate passed")


In [ ]:
st, (out, have) = list(shown.items())[0]
fig, ax = plt.subplots(2, len(CELL_NAMES), figsize=(3.0*len(CELL_NAMES), 6.2))
for i, c in enumerate(CELL_NAMES):
    v = out[c]; z = v.shape[0]//2
    ax[0,i].imshow(v[z], cmap="gray", vmin=0, vmax=255); ax[0,i].axis("off")
    ax[0,i].set_title(f"{c}\navail={have[c]}  z={z}", fontsize=8)
    ax[1,i].imshow(v[max(0,z-5)], cmap="gray", vmin=0, vmax=255); ax[1,i].axis("off")
lh = LAT.set_index("StudyInstanceUID").loc[st]
fig.suptitle(f"{st[:24]}...  laterality={lh['laterality']} ({lh['lat_source']})  |  "
             f"{CFG['OUT_SIZE']}px, {CFG['FOV_MM']/CFG['OUT_SIZE']:.3f} mm/px", fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# Mirroring test: mean coronal of originally-left vs originally-right knees.
# If the convention is inverted these come out as mirror images of each other.
acc = {"L": [], "R": []}
for j in JOBS[:120]:
    s = j[0]; side = lat_map.get(s, "U")
    if side not in acc or len(acc[side]) >= 12: continue
    if "COR_FS" not in j[1]: continue
    try:
        v, _ = load_cell(ROOT_TR, s, j[1]["COR_FS"], "COR_FS", side)
        acc[side].append(v[v.shape[0]//2].astype(np.float32))
    except Exception:
        pass
    if len(acc["L"]) >= 12 and len(acc["R"]) >= 12: break

if acc["L"] and acc["R"]:
    mL, mR = np.mean(acc["L"], 0), np.mean(acc["R"], 0)
    fig, ax = plt.subplots(1, 3, figsize=(11, 4))
    ax[0].imshow(mL, cmap="gray"); ax[0].set_title(f"mean COR_FS, originally LEFT (n={len(acc['L'])})"); ax[0].axis("off")
    ax[1].imshow(mR, cmap="gray"); ax[1].set_title(f"mean COR_FS, originally RIGHT (n={len(acc['R'])})"); ax[1].axis("off")
    ax[2].imshow(np.abs(mL-mR), cmap="magma"); ax[2].set_title("|difference|"); ax[2].axis("off")
    plt.tight_layout(); plt.show()
    d_same, d_flip = float(np.abs(mL-mR).mean()), float(np.abs(mL-mR[:, ::-1]).mean())
    print(f"mean |L-R| = {d_same:.2f} | mean |L-flip(R)| = {d_flip:.2f}")
    print("=> mirroring is CORRECT" if d_same < d_flip else
          "=> WARNING: flipping R matches better, the convention is inverted")


## 6. Full run


In [ ]:
todo = [j for j in JOBS if not (OUT/"cache"/f"{j[0]}.npz").exists()]
print(f"{len(JOBS)-len(todo)} already cached, {len(todo)} to do")

rows_meta, rows_cell, failures = [], [], []
t0 = time.time()
with ProcessPoolExecutor(max_workers=CFG["N_WORKERS"]) as ex:
    for i, (st, out, meta, have, errs) in enumerate(ex.map(process_study, todo, chunksize=4)):
        try: sz = save_study(st, out, have)
        except Exception as e:
            failures.append((st, "save", str(e)[:120])); continue
        rows_meta.append(dict(StudyInstanceUID=st, npz_bytes=sz,
                              **{f"avail_{c}": have[c] for c in CELL_NAMES}))
        for c, m in meta.items(): rows_cell.append(dict(StudyInstanceUID=st, **m))
        for c, e in errs.items(): failures.append((st, c, e))
        if (i+1) % 250 == 0:
            el = time.time()-t0
            print(f"  {i+1}/{len(todo)}  {el/60:.1f} min  eta {el/(i+1)*(len(todo)-i-1)/60:.1f} min")
print(f"\ndone in {(time.time()-t0)/60:.1f} min | failures: {len(failures)}")


In [ ]:
META = pd.DataFrame(rows_meta); CELLM = pd.DataFrame(rows_cell)

STUDY = LAT[["StudyInstanceUID","laterality","lat_source","lat_tag","lat_report","centre_x"]].merge(
    META, on="StudyInstanceUID", how="right")
STUDY = STUDY.merge(REP[["StudyInstanceUID","lang","n_chars"]], on="StudyInstanceUID", how="left")
if "PatientSex" in train.columns:
    STUDY = STUDY.merge(train[["StudyInstanceUID","PatientSex"]], on="StudyInstanceUID", how="left")

lab_mask = train[LABELS].notna().all(axis=1)
STUDY = STUDY.merge(train.loc[lab_mask, ["StudyInstanceUID"]+LABELS],
                    on="StudyInstanceUID", how="left")
STUDY["has_gold"] = STUDY[LABELS].notna().all(axis=1).astype(int)

print("cached studies:", len(STUDY), "| with gold labels:", int(STUDY["has_gold"].sum()))
print("\ncell availability (0=absent, 1=real, 2=fallback-filled):")
print(STUDY[[f"avail_{c}" for c in CELL_NAMES]].apply(pd.Series.value_counts).fillna(0).astype(int))
print(f"\ntotal cache: {STUDY['npz_bytes'].sum()/1e9:.2f} GB "
      f"({STUDY['npz_bytes'].mean()/1e6:.2f} MB/study)")

STUDY.to_parquet(OUT/"study_meta.parquet", index=False)
REP.to_parquet(OUT/"reports_clean.parquet", index=False)
CELLM.to_parquet(OUT/"series_index.parquet", index=False)
if failures:
    pd.DataFrame(failures, columns=["StudyInstanceUID","cell","error"]).to_csv(OUT/"failures.csv", index=False)

with open(OUT/"preprocess_config.json", "w") as f:
    json.dump({**CFG, "cells": CELL_NAMES, "mm_per_px": CFG["FOV_MM"]/CFG["OUT_SIZE"],
               "geom_thresh_accuracy": float(best_acc),
               "generated": time.strftime("%Y-%m-%d %H:%M")}, f, indent=2, default=str)
print("\nwrote:", [p.name for p in sorted(OUT.iterdir())])


---

**Check before moving on**

1. §5's size gate passed and the total in §6 is comfortably under 20 GB.
2. §5's mirroring test prints `mirroring is CORRECT`.
3. Cell availability looks like your previous run — `COR_T1` will still show a large number of
   fallback-filled (`avail = 2`) entries, which is expected.

Save this notebook's output and attach it to the training notebook. `preprocess_config.json` now records
`OUT_SIZE = 224`, and both downstream notebooks read the geometry from it rather than hard-coding it.
